# MCA Timestamp Example

Continuous event logging with microsecond timestamps, plus live histogram.

Uses the `mca_timestamp_1ch` bitfile: IIR highpass -> FIR -> peak detector -> histogram + event logger.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
from redpitaya_control.redpitaya_dev import redpitaya_dev
from redpitaya_control.event_logger import EventLogger, unpack, load_run, fit_clock, counter_to_unix
from redpitaya_control import compute_coeff

## 1. Connect and configure

In [ ]:
RP_HOST = os.environ.get("RP_HOST", "171.64.56.120")
dev = redpitaya_dev(RP_HOST, "config/mca_timestamp_1ch.json")
dev.base.load_bitfile()

# Signal chain
dev.set_all_registers('iir1', compute_coeff.highpass_1st(1e4, Ts=16e-9), reset=True)
dev.set_register('fir9', 'h0', 0.99)

# Peak detector
dev.set_register("peak_detector", "invert_input", 0)
dev.set_register("peak_detector", "trig_level", 0.01)
dev.set_register("peak_detector", "integration_mode", 0)
dev.set_register("peak_detector", "n_integration", 1000)

# Histogram
dev.set_register("histogram", "offset", 0)
dev.set_register("histogram", "gain", 0.1)
dev.set_register("histogram", "band_low", 0.0)
dev.set_register("histogram", "band_high", 1.0)
dev.set_register("histogram", "pulse_width", 1024)
dev.set_register("histogram", "clear_bins", 1)
time.sleep(0.01)
dev.set_register("histogram", "clear_bins", 0)
dev.set_register("histogram", "counting_enable", 1)

## 2. Run the event logger

`veto_ms=100` tags every event within 100 ms of a channel-B transition with
record bit 63 (rather than dropping it), so it can be filtered offline. Use
`veto_ms=0` to disable the tag, or `duration=None` to run until Ctrl+C.

In [ ]:
log = EventLogger(dev)
log.configure(flush_ms=100, veto_ms=100)   # 100 ms chB-transition veto
log.run(duration=60, output_dir="run1")

## 3. Load and inspect

In [ ]:
raw = load_run("run1/events_*.bin")
ts_us, energy, chb, veto = unpack(raw)

n, n_veto = len(veto), int(veto.sum())
print(f"{n} events, {n_veto} vetoed ({100*n_veto/max(n,1):.1f}%) near chB transitions")
if n > 1:
    print(f"span {(ts_us[-1]-ts_us[0])/1e6:.1f} s")

keep = veto == 0   # boolean mask -- apply to any per-event array below

## 4. Clock calibration

In [ ]:
a, b, ppm = fit_clock("run1/tiepoints.csv")
print(f"Crystal drift: {ppm:+.1f} ppm")
t_unix = counter_to_unix(ts_us, a, b)

## 5. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Energy spectrum (chB-transition events removed)
axes[0].hist(energy[keep], bins=256, range=(0, 2**15), log=True, color='steelblue', edgecolor='none')
axes[0].set_xlabel("Energy (ADC units)")
axes[0].set_ylabel("Counts")
axes[0].set_title(f"Energy spectrum ({int(keep.sum())} kept, {int((~keep).sum())} vetoed)")

# Event rate: kept vs vetoed
if len(t_unix) > 10:
    t_rel = t_unix - t_unix[0]
    bins = np.arange(0, t_rel[-1] + 1, 1.0)
    rate_keep, edges = np.histogram(t_rel[keep], bins=bins)
    rate_veto, _     = np.histogram(t_rel[~keep], bins=bins)
    axes[1].step(edges[:-1], rate_keep, where='post', color='darkorange', label='kept')
    axes[1].step(edges[:-1], rate_veto, where='post', color='gray',       label='vetoed')
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Events / s")
    axes[1].set_title("Event rate")
    axes[1].legend()

plt.tight_layout()
plt.show()